In [ ]:
import numpy as np
from scipy.optimize import least_squares

from Options_Math_Helpers import *

In [ ]:
class SVISkew(OptionsMathHelpers):

    def __init__(self):
        # constructor
        pass

    # -------------------------
    # Core SVI parameterization
    # -------------------------

    def _log_moneyness(self, fwd, strike):
        return np.log(strike / fwd)

    def svi_total_variance(self, fwd=None, strike=None, time=None,
                           a=None, b=None, rho=None, m=None, sigma=None,
                           eps=1e-12, **kwargs):
        """
        Raw SVI total variance w(k)
        """
        k = self._log_moneyness(fwd, strike)

        diff = k - m
        sqrt_term = np.sqrt(diff * diff + sigma * sigma)

        w = a + b * (rho * diff + sqrt_term)

        # Enforce positivity for numerical stability
        return np.maximum(w, eps)

    def svi_vol(self, fwd=None, strike=None, time=None,
                a=None, b=None, rho=None, m=None, sigma=None,
                **kwargs):
        """
        Implied volatility from SVI
        """
        fwd, strike, time = self.to_arrays(fwd, strike, time)

        w = self.svi_total_variance(fwd=fwd,
                                    strike=strike,
                                    time=time,
                                    a=a, b=b, rho=rho, m=m, sigma=sigma)

        return np.sqrt(w / time)

    # -------------------------
    # Calibration helpers
    # -------------------------

    def _svi_vol_weighted(self, params, fwd, strike, time, target_vols, weights):
        a, b, rho, m, sigma = params

        model_vols = self.svi_vol(fwd=fwd,
                                  strike=strike,
                                  time=time,
                                  a=a, b=b, rho=rho, m=m, sigma=sigma)

        vol_errors = model_vols - target_vols
        return weights * vol_errors

    def calibrate_svi_weighted(self,
                               fwd=None, strike=None, time=None,
                               target_vols=None,
                               weighting='vega',
                               weights=None,
                               weight_eps=1e-8,
                               initial_guess=(0.01, 0.1, 0.0, 0.0, 0.1),
                               bounds=([0.0, 1e-6, -0.999, -5.0, 1e-6],
                                       [5.0, 5.0, 0.999, 5.0, 5.0]),
                               **lsq_kwargs):
        """
        weighting: 'vega' | 'sqrt' | 'norm'
        """

        fwd, strike, time, target_vols, weights, weight_eps, initial_guess = \
            self.to_arrays(fwd, strike, time, target_vols,
                           weights, weight_eps, initial_guess)

        target_vols = np.nan_to_num(target_vols, nan=weight_eps)

        if weighting in ('vega', 'sqrt', 'norm'):
            weights = np.nan_to_num(weights, nan=weight_eps)
            weights = np.maximum(weights, weight_eps)

            if weighting == 'vega':
                weights = weights
            elif weighting == 'sqrt':
                weights = np.sqrt(weights)
            else:  # 'norm'
                weights = weights / np.max(weights)
        else:
            raise ValueError("Unknown weighting: choose 'vega', 'sqrt', or 'norm'")

        obj_fn = lambda params: self._svi_vol_weighted(params,
                                                       fwd, strike, time,
                                                       target_vols, weights)

        result = least_squares(obj_fn,
                               initial_guess,
                               bounds=bounds,
                               **lsq_kwargs)

        a, b, rho, m, sigma = result.x

        return a, b, rho, m, sigma, result